In [1]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle


PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from utils.hospital_env import HospitalEnv


data = pd.read_csv(
    os.path.join(PROJECT_ROOT, "data", "synthetic_hospital_data.csv")
)


arrival_bins = np.linspace(0, 480, 6)
slot_bins = np.linspace(0, 480, 6)
no_show_bins = np.linspace(0, 1, 6)
icu_bins = np.linspace(0, 1, 6)

def discretize_state(state):
    arrival, slot, priority, no_show, icu_ratio = state
    a = np.digitize(arrival, arrival_bins)
    s = np.digitize(slot, slot_bins)
    n = np.digitize(no_show, no_show_bins)
    i = np.digitize(icu_ratio, icu_bins)
    return (a, s, int(priority), n, i)


actions = [0, 1]

def get_Q(Q, state):
    if state not in Q:
        Q[state] = {a: 0.0 for a in actions}
    return Q[state]


alpha = 0.1
gamma = 0.9
epsilon_start = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995
episodes = 300
runs = 3

all_rewards = []
all_waiting = []
all_util = []


for run in range(runs):
    print(f"\nRun {run+1}/{runs}")

    Q = {}
    epsilon = epsilon_start

    episode_rewards = []
    episode_waiting = []
    episode_util = []

    for ep in range(episodes):

        env = HospitalEnv(data)
        state = discretize_state(env.reset())
        done = False

        total_reward = 0
        total_wait = 0
        total_util = 0
        steps = 0

        while not done:

            # ε-greedy policy
            if np.random.rand() < epsilon:
                action = np.random.choice(actions)
            else:
                action = max(get_Q(Q, state), key=get_Q(Q, state).get)

            next_state, reward, done, info = env.step(action)

            total_reward += reward
            total_wait += info["waiting_time"]
            total_util += info["resource_util"]
            steps += 1

            if not done:
                next_state = discretize_state(next_state)
                best_next = max(get_Q(Q, next_state).values())
            else:
                best_next = 0

            # Q-learning update
            get_Q(Q, state)[action] += alpha * (
                reward + gamma * best_next - get_Q(Q, state)[action]
            )

            state = next_state

        epsilon = max(epsilon_min, epsilon * epsilon_decay)

        episode_rewards.append(total_reward)
        episode_waiting.append(total_wait / max(steps, 1))
        episode_util.append(total_util / max(steps, 1))

        print(f"Episode {ep+1}: Reward = {total_reward}")

    all_rewards.append(episode_rewards)
    all_waiting.append(episode_waiting)
    all_util.append(episode_util)

print("\nTraining completed.")


avg_rewards = np.mean(all_rewards, axis=0)
avg_waiting = np.mean(all_waiting, axis=0)
avg_util = np.mean(all_util, axis=0)


results = pd.DataFrame({
    "episode": range(1, episodes + 1),
    "reward": avg_rewards,
    "waiting_time": avg_waiting,
    "resource_util": avg_util
})

results.to_csv("tabular_training_log.csv", index=False)


summary = pd.DataFrame({
    "avg_reward": [np.mean(avg_rewards[-50:])],
    "avg_waiting_time": [np.mean(avg_waiting[-50:])],
    "avg_resource_util": [np.mean(avg_util[-50:])]
})

summary.to_csv("tabular_summary.csv", index=False)


def convergence_episode(rewards):
    for i in range(50, len(rewards)):
        if abs(rewards[i] - rewards[i-10]) < 1:
            return i
    return len(rewards)

conv_ep = convergence_episode(avg_rewards)

conv_df = pd.DataFrame({"convergence_episode": [conv_ep]})
conv_df.to_csv("tabular_convergence.csv", index=False)


with open("tabular_q_table.pkl", "wb") as f:
    pickle.dump(Q, f)


window = 10
smoothed_rewards = pd.Series(avg_rewards).rolling(window).mean()

plt.figure()
plt.plot(smoothed_rewards)
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("Tabular Q-Learning Training Curve")
plt.savefig("tabular_reward_curve.png")
plt.close()

print("\nFiles saved:")
print("tabular_training_log.csv")
print("tabular_summary.csv")
print("tabular_convergence.csv")
print("tabular_reward_curve.png")
print("tabular_q_table.pkl")


Run 1/3
Episode 1: Reward = -2054
Episode 2: Reward = -2441
Episode 3: Reward = -1971
Episode 4: Reward = -2126
Episode 5: Reward = -1797
Episode 6: Reward = -1556
Episode 7: Reward = -1242
Episode 8: Reward = -1905
Episode 9: Reward = -1925
Episode 10: Reward = -1574
Episode 11: Reward = -2146
Episode 12: Reward = -1726
Episode 13: Reward = -1449
Episode 14: Reward = -1846
Episode 15: Reward = -1825
Episode 16: Reward = -1656
Episode 17: Reward = -1870
Episode 18: Reward = -2019
Episode 19: Reward = -1277
Episode 20: Reward = -2096
Episode 21: Reward = -1637
Episode 22: Reward = -1849
Episode 23: Reward = -1501
Episode 24: Reward = -1980
Episode 25: Reward = -1792
Episode 26: Reward = -1327
Episode 27: Reward = -1894
Episode 28: Reward = -1834
Episode 29: Reward = -1712
Episode 30: Reward = -1733
Episode 31: Reward = -1517
Episode 32: Reward = -1492
Episode 33: Reward = -1672
Episode 34: Reward = -1696
Episode 35: Reward = -1976
Episode 36: Reward = -1833
Episode 37: Reward = -1295
E

In [2]:
import pandas as pd

summary = pd.read_csv("tabular_summary.csv")
conv = pd.read_csv("tabular_convergence.csv")

performance_row = pd.DataFrame({
    "Algorithm": ["Tabular Q-Learning"],
    "Avg Reward": [summary["avg_reward"][0]],
    "Waiting Time": [summary["avg_waiting_time"][0]],
    "Resource Utilization": [summary["avg_resource_util"][0]],
    "Convergence Episode": [conv["convergence_episode"][0]]
})

performance_row.to_csv("tabular_q_performance_row.csv", index=False)

print("Tabular Q performance row created.")
print(performance_row)

Tabular Q performance row created.
            Algorithm  Avg Reward  Waiting Time  Resource Utilization  \
0  Tabular Q-Learning -505.533333        54.534             90.313333   

   Convergence Episode  
0                  300  
